# Chapter 4 &mdash; DFA Everywhere

**Concept 1 of the Chapter 4 decomposition:** *DFA Everywhere: Why Finite-State Machines Matter*

Lexers, traffic lights, pilot mode confusion, deep packet inspection &mdash; finite-state machines are already everywhere.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-DFA-Everywhere/Concept-DFA-Everywhere.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Finite-state machines run **lexers** in compilers, **traffic lights**, **airplane
control panels** (and the pilot's own mental model &mdash; a mismatch is called *mode
confusion*), and **deep packet inspection** hardware.

Speed matters there for a security reason: a front-end machine too slow to keep up
invites a **denial-of-service** attack.

Variants you will meet elsewhere &mdash; Mealy machines, communicating FSMs, UML
statecharts, B&uuml;chi automata &mdash; are all variations on what we study here.

## 2. Definitions

### A traffic light as a DFA

Four states, cycling on a `t` (timer) event, with a `p` (pedestrian button).

In [ ]:
light = md2mc('''DFA
IF   : t -> Grn
IF   : p -> IF
Grn  : t -> Yel
Grn  : p -> Grn
Yel  : t -> IF
Yel  : p -> Yel
''')
print("states :", sorted(light["Q"]))
assert light["q0"] == "IF"

### A malware-signature scanner

One pass, constant memory, never backs up.

In [ ]:
dpi = md2mc('''DFA
I    : 1 -> S1
I    : 0 -> I
S1   : 1 -> S11
S1   : 0 -> I
S11  : 1 -> S11
S11  : 0 -> S110
S110 : 1 -> F
S110 : 0 -> I
F    : 0 | 1 -> F
''')
print("signature scanner states :", sorted(dpi["Q"]))

## 3. Tests

The light cycles forever and never grows.

In [ ]:
seq = 'ttt' * 3
print("after", seq, "-> accepted?", accepts_dfa(light, seq))
assert accepts_dfa(light, 'ttt')    # back to the start state
print("states used: %d, however long the input" % len(light["Q"]))

The scanner finds the signature in one pass.

In [ ]:
for pkt in ['0001101000', '1111', '1101', '0011010']:
    print("%-12s contains 1101? %s" % (pkt, accepts_dfa(dpi, pkt)))
assert accepts_dfa(dpi, '1101') and not accepts_dfa(dpi, '1111')

## 4. Animation

Watch the scanner fall back correctly on a mismatch &mdash; that is what lets it make one pass.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(dpi, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Add an `emergency` input to the traffic light that jumps straight to red.
2. Change the signature to `1011`. Which fall-back edges change?
3. Why would a *slow* scanner be a security problem and not just an annoyance?

In [ ]:
# Your work for the exercises above.